# Baseline Logistic Regression — gambar laporan

Notebook ini hanya membaca artefak dari `data/gold/models/lr-t3/`.
Tidak ada training dan tidak ada pembacaan parquet Gold.
Jalankan `task ml-train -- --model lr --features t3` lebih dulu.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL = ROOT / "data" / "gold" / "models" / "lr-t3"

metrics = json.loads((MODEL / "metrics.json").read_text(encoding="utf-8"))
calibration = pd.DataFrame(
    json.loads((MODEL / "calibration.json").read_text(encoding="utf-8"))
)
coefficients = pd.read_csv(MODEL / "coefficients.csv")

metrics["test"]

In [ ]:
rows = []
for split in ("valid", "test"):
    m = metrics[split]
    rows.append(
        {
            "split": split,
            "PR-AUC": m["pr_auc"],
            "ROC-AUC": m["roc_auc"],
            "Brier": m["brier"],
            "F1": m["f1"],
            "Precision": m["precision"],
            "Recall": m["recall"],
        }
    )
pd.DataFrame(rows).round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], "--", color="grey", label="sempurna")
ax.plot(calibration["mean_predicted"], calibration["observed"], marker="o", label="LR")
ax.set_xlabel("probabilitas prediksi")
ax.set_ylabel("fraksi observasi")
ax.set_title("Kalibrasi test")
ax.legend()
plt.tight_layout()
plt.savefig(MODEL / "calibration.png", dpi=150)
plt.show()

In [ ]:
top = coefficients.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top["feature"], top["odds_ratio"])
ax.axvline(1.0, color="grey", linestyle="--")
ax.set_xlabel("odds ratio")
ax.set_title("15 koefisien dengan pengaruh terbesar")
plt.tight_layout()
plt.savefig(MODEL / "coefficients.png", dpi=150)
plt.show()